# Predicción de subscripción a un producto bancario

## 1. EDA

Para hacer un EDA, debemos usar pickle para deserializar los datos.

### 1.1. Instalación de pickle y pandas

In [2]:
%pip install pandas 

  Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ----------------------------------- ---- 8.7/9.9 MB 54.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 50.3 MB/s  0:00:00
Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl (12.4 MB)

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------

### 1.2. Deserialización de los datos

Con la librería pickle, extraemos los datos de forma sencilla. 

In [3]:
import pickle as pkl
import os

file = "./dataset/bank_10.pkl"

if os.path.exists(file):
    with open(file, 'rb') as fd:
        df = pkl.load(fd)
        print(df)


       age          job  marital  education default  balance housing loan  \
0       59       admin.  married  secondary      no     2343     yes   no   
1       56         None  married  secondary      no       45      no   no   
2       41   technician  married  secondary      no     1270     yes   no   
3       55         None  married  secondary      no     2476     yes   no   
4       54       admin.     None   tertiary      no      184      no   no   
...    ...          ...      ...        ...     ...      ...     ...  ...   
11157   33  blue-collar   single    primary      no        1     yes   no   
11158   39     services  married  secondary      no      733      no   no   
11159   32   technician   single  secondary      no       29      no   no   
11160   43   technician  married  secondary      no        0      no  yes   
11161   34   technician  married  secondary      no        0      no   no   

        contact  day month  duration  campaign  pdays  previous poutcome  \

C:\Users\Sergio\AppData\Local\Temp\ipykernel_4764\713815639.py:8: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  df = pkl.load(fd)


### 1.3. Análisis de variables e instancias

A continuación, analizaremos tanto el número de variables y de instancias, así como sus principales características. 

En cuanto al número de variables y de instancias se puede observar fácilmente de la siguiente forma. 

In [4]:
n_vars = len(df.columns) # Número de variables
n_instances = len(df) # Número de instancias

print(f"Número de variables: {n_vars}\nNúmero de instancias: {n_instances}")


Número de variables: 17
Número de instancias: 11000


Antes de nada, para evitar futuros problemas, reemplazaremos posibles textos que puedan ser malinterpretados por Nan. 

In [15]:
import numpy as np

df = df.replace(['None', 'null', 'NaN', 'N/A', ''], np.nan)

Sabiendo que el número de variables es 17, nos gustaría saber cuántas de ellas son numéricas, cuántas son categóricas y cuántas son ordinales. 

In [ ]:
num_vars = df.select_dtypes(include=['number']).columns.tolist() # Variables numéricas

cat_ord_vars = df.select_dtypes(include=['object', 'category']) # Variables categóricas y ordinales

# Ahora debemos distinguir cuáles son categóricas y cuáles son ordinales
for var in cat_ord_vars.columns:
    print(f"Variable {var} tiene los valores {df[var].unique()}")


Variable job tiene los valores ['admin.' None 'technician' 'management' 'retired' 'services'
 'blue-collar' 'unemployed' 'entrepreneur' 'housemaid' 'unknown'
 'self-employed' 'student']
Variable marital tiene los valores ['married' None 'single' 'divorced']
Variable education tiene los valores ['secondary' 'tertiary' 'primary' 'unknown']
Variable default tiene los valores ['no' 'yes']
Variable housing tiene los valores ['yes' 'no']
Variable loan tiene los valores ['no' 'yes']
Variable contact tiene los valores ['unknown' 'cellular' 'telephone']
Variable month tiene los valores ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr' 'sep']
Variable poutcome tiene los valores ['unknown' 'other' 'failure' 'success']
Variable deposit tiene los valores ['yes' 'no']
Variables ordinales detectadas: Ninguna


Viendo los valores, podemos entonces ver que hay algunas de ellas que mantienen un orden, y por tanto, son ordinales. 

Estas variables son:

Education

Month

Por tanto, el resto serán categóricas

In [6]:
cat_vars = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'poutcome', 'deposit']
ord_vars = ['education', 'month']


Ahora que ya hemos distinguido entre variables categóricas y ordinales, debemos analizar su cardinalidad

In [7]:
cat_hcard_var = [] # Variables categóricas con alta cardinalidad
ord_hcard_var = [] # Variables ordinales con alta cardinalidad

# Vamos a definir una variable de alta cardinalidad cuando hay más de 10 valores únicos
for var in cat_vars:
    if df[var].nunique() > 10:
        cat_hcard_var.append(var)

for var in ord_vars:
    if df[var].nunique() > 10:
        cat_hcard_var.append(var)

print(f"Variables categóricas con alta cardinalidad: {", ".join(cat_hcard_var) if cat_hcard_var else "No se encontraron elementos"}")
print(f"Variables ordinales con alta cardinalidad: {", ".join(ord_hcard_var) if ord_hcard_var else "No se encontraron elementos"}")

Variables categóricas con alta cardinalidad: job, month
Variables ordinales con alta cardinalidad: No se encontraron elementos


Por tanto, a modo de resumen, tenemos

In [8]:
print(f"Variables númericas: {", ".join(num_vars) if num_vars else "No se encontraron elementos"}")
print("_________________________________________________________________________________________")
print(f"Variables categóricas: {", ".join(cat_vars) if cat_vars else "No se encontraron elementos"}")
print(f"\tDe las cuales con alta cardinalidad: {", ".join(cat_hcard_var) if cat_hcard_var else "No se encontraron elementos"}")
print("_________________________________________________________________________________________")
print(f"Variables númericas: {", ".join(ord_vars) if ord_vars else "No se encontraron elementos"}")
print(f"\tDe las cuales con alta cardinalidad: {", ".join(ord_hcard_var) if ord_hcard_var else "No se encontraron elementos"}")

Variables númericas: age, balance, day, duration, campaign, pdays, previous
_________________________________________________________________________________________
Variables categóricas: job, marital, default, housing, loan, contact, poutcome, deposit
	De las cuales con alta cardinalidad: job, month
_________________________________________________________________________________________
Variables númericas: education, month
	De las cuales con alta cardinalidad: No se encontraron elementos


Ahora que tenemos una idea más clara de cómo son las variables, debemos analizar cuales de ellas tienen valores faltantes y, en ese caso, cuánta cantidad de ellos

In [9]:
# Creamos una lista de tuplas donde cada elemento es una tupla de dos elementos
# Primer elemento: variable
# Segundo elemento: número de valores faltantes
missing_values_count = []

for col in df.columns:
    n_null = df[col].isnull().sum() # Contamos el número de valores faltantes
    missing_values_count.append((col, int(n_null)))

    if n_null > 0:
        print(f"La variable {col} tiene {n_null} valores faltantes")
    else:
        print(f"La variable {col} no tiene valores faltantes")

La variable age no tiene valores faltantes
La variable job tiene 332 valores faltantes
La variable marital tiene 183 valores faltantes
La variable education no tiene valores faltantes
La variable default no tiene valores faltantes
La variable balance no tiene valores faltantes
La variable housing no tiene valores faltantes
La variable loan no tiene valores faltantes
La variable contact no tiene valores faltantes
La variable day no tiene valores faltantes
La variable month no tiene valores faltantes
La variable duration no tiene valores faltantes
La variable campaign no tiene valores faltantes
La variable pdays no tiene valores faltantes
La variable previous no tiene valores faltantes
La variable poutcome no tiene valores faltantes
La variable deposit no tiene valores faltantes


Debemos analizar también si tenemos columnas constantes o no. Para ello, vemos la cantidad de valores únicos que tiene cada variable. 
Las variables con valores constantes tienen, por lo tanto, varianza 0, y como consecuencia, no aportan información útil y tan solo ocupan espacio innecesario. 

In [16]:
# Es seguro porque ya hemos convertido posibles textos como "None" "null" o "NaN" a NaN de numpy
constants = [col for col in df.columns if df[col].nunique() <= 1]

print(f"Las variables con valores constantes son: {", ".join(constants) if constants else "No hay variables con valores constantes"}")

Las variables con valores constantes son: No hay variables con valores constantes


Por motivos similares a la anterior, debemos analizar si hay alguna variable con IDs, pues al ser distinto para cada instancia, tampoco aporta información útil. 

In [17]:
ids = [col for col in df.columns if df[col].nunique() == len(df)]

print(f"Las variables que son IDs son: {", ".join(ids) if ids else "No se encontraron IDs"}")

Las variables que son IDs son: No se encontraron IDs


### 1.X. Visualización de los datos

Para poder visualizar mejor los datos y hacer un análisis de ellos, usaremos streamlit.

In [21]:
%pip install streamlit

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached charset_normalizer-3.4.4-cp314-cp314-win_amd64.whl.metadata (38 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-win_amd64.whl.metadata (2.8 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   --------------------------------- ------ 7.6/9.1 MB 42.3 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 40.6 MB/s  0:00:00
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 795.4/795.4 kB 36.3 MB/s  0:00:00
   --------------

  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 32] El proceso no tiene acceso al archivo porque está siendo utilizado por otro proceso: 'd:\\UC3M\\3° CARRERA\\2 C UATRIMESTRE\\MACHINE LEARNING\\Practica1-ML\\.venv\\Lib\\site-packages\\streamlit\\web\\server\\app_static_file_handler.py'
Check the permissions.



  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached gitdb-4.0.12-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
Using cached streamlit-1.55.0-py3-none-any.whl (9.1 MB)
Using cached altair-6.0.0-py3-none-any.whl (795 kB)
Using cached gitpython-3.1.46-py3-none-any.whl (208 kB)
Using cached gitdb-4.0.12-py3-none-any.whl (62 kB)
Using cached pydeck-0.9.1-py2.py3-none-any.whl (6.9 MB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)

   ---

De esta forma, podremos ver de forma más visual algunas propiedades de los datos.

In [ ]:
%%writefile visualizer.py
import pandas as pd
import streamlit as st

import pickle as pkl

file = "dataset/bank_10.pkl"

with open(file, 'rb') as fd:
    df = pkl.load(fd)
    print(df)

# Extract the number of variables and instances
df_shape = df.shape

# Create a chart
st.subheader("Número de instancias y variables")
st.table(df_shape, columns=["Instancias", "Variables"])


Overwriting visualizer.py


### 1.4 Correr la aplicación

Para visualizar los datos, ejecutamos el siguiente comando:

In [ ]:
# No termina la ejecución porque streamlit se queda ejecutando infinitamente
!streamlit run visualizer.py

^C
